In [ ]:
# bootstrap: Colab clone + local import of `gpurt` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "02-cuda-nccl-runtime/cuda-and-nccl/cuda-nccl-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "gpurt").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 04 · busbw and the α-β fit: reading nccl-tests like a performance engineer

**Tier:** T0 — works on the bundled nccl-tests sample (**illustrative**: generated by
`tools/make_fixtures.py` from a stated α-β model, not measured on hardware) and on this machine's own
sweep from notebook 03's backend. Bring your own logs from a GPU box (`deploy/any-gpu/run_nccl_tests.sh`,
`deploy/gke` Job) and the same cells read them — that is the T1/T2 use.

## The one-minute version

* Every column of nccl-tests can be recomputed: **algbw = size / time**, **busbw = algbw × factor(op, n)**.
* A size sweep has two regimes: a **latency floor** (time flat in size) and a **bandwidth plateau**
  (busbw flat). The α-β fit `t = α + S/B` gives both, and the **half-bandwidth size S½ = α·B** separates them.
* The plateau busbw is held against the **link**: NVLink hundreds of GB/s per GPU, PCIe Gen4 x16 about
  25 GB/s in practice. Far below the link means NCCL is not using the path you think.
* Inference: decode's tensor-parallel all-reduces sit on the latency floor — α, not bandwidth, sets their
  cost; prefill's sit on the plateau.

Concepts: [the primer](../../PRIMER.md) §5 *Collectives* (α-β costs, algbw and busbw, latency- vs
bandwidth-bound messages).

In [ ]:
from pathlib import Path

from gpurt import nccltests
from gpurt.dist import alphabeta as ab
from gpurt.dist import busbw as bw

FIX = Path(nccltests.__file__).parent / "fixtures"
text = (FIX / "nccl_all_reduce_8gpu_sample.txt").read_text()
print("\n".join(text.splitlines()[:2]))  # the provenance header: illustrative, and the generating model
res = nccltests.parse(text, op="all_reduce")
print(f"\nop={res.op} ranks={res.nranks} rows={len(res.rows)} header={res.header}")
for r in res.rows[:3] + res.rows[-3:]:
    print(f"{r.size:>12} B  {r.time_us:>9.2f} µs  algbw {r.algbw:7.2f}  busbw {r.busbw:7.2f} GB/s")

## Exercise 4.1 — recompute the columns

Write `recompute(size_bytes, time_us, op, n)` returning `(algbw, busbw)` in GB/s (1e9). The check
compares with every printed row (the printed values are rounded to two decimals).

In [ ]:
def recompute(size_bytes: int, time_us: float, op: str, n: int) -> tuple[float, float]:
    algbw = size_bytes / (time_us * 1e-6) / 1e9
    return algbw, algbw * bw.bus_factor(op, n)

In [ ]:
for r in res.rows:
    a, b = recompute(r.size, r.time_us, "all_reduce", res.nranks)
    assert abs(a - r.algbw) <= max(0.011, 0.02 * a) and abs(b - r.busbw) <= max(0.011, 0.02 * b), r
print(f"✅ every row: busbw = algbw x 2(n-1)/n = algbw x {bw.bus_factor('all_reduce', res.nranks)} for n = {res.nranks}")

## Seeing the two regimes

busbw against size, as a text chart. Small messages barely register; the curve bends around S½ and
flattens at the link-limited plateau.

In [ ]:
peak = max(r.busbw for r in res.rows)
for r in res.rows[::2]:
    print(f"{r.size:>12} B |{'#' * int(50 * r.busbw / peak):<50}| {r.busbw:7.2f} GB/s")

## Exercise 4.2 — fit α and B yourself

Fit `t = α + S/B` to (size, time) by least squares on **relative** error — minimise
Σ((α + β·S − t)/t)² with β = 1/B — so that a 20 µs point and a 5 ms point weigh the same. Divide each
row of the linear system `[1, S]·[α, β] = t` by `t` and use `np.linalg.lstsq`. Return `(alpha_s, bw_Bps)`.

In [ ]:
import numpy as np  # noqa: E402


def fit_alpha_beta(sizes, times_s) -> tuple[float, float]:
    S, t = np.asarray(sizes, float), np.asarray(times_s, float)
    X = np.column_stack([1 / t, S / t])
    (alpha, beta), *_ = np.linalg.lstsq(X, np.ones_like(t), rcond=None)
    return float(alpha), float(1 / beta)

In [ ]:
sizes = [r.size for r in res.rows]
times = [r.time_us * 1e-6 for r in res.rows]
alpha, B = fit_alpha_beta(sizes, times)
lib = ab.fit(sizes, times)
assert abs(alpha - lib.alpha_s) <= 1e-9 * lib.alpha_s + 1e-12 and abs(B - lib.bw_Bps) <= 1e-6 * lib.bw_Bps
assert abs(alpha - 20e-6) < 2e-6  # the sample's header says it was generated with alpha = 20 us
assert abs(B * bw.bus_factor("all_reduce", 8) / 1e9 - 400) < 20  # ... and busbw -> 400 GB/s
print(f"✅ α = {alpha * 1e6:.1f} µs, algbw -> {B / 1e9:.1f} GB/s, busbw -> {B * 1.75 / 1e9:.0f} GB/s "
      "(recovered the model behind the illustrative sample)")

## Exercise 4.3 — the half-bandwidth size, two ways

The fit's α is the **whole collective's** latency, and B its asymptotic **algbw**. The primer writes
the same ring with a per-hop latency α_hop and per-link bandwidth B_link:
`t = 2(n−1)·α_hop + 2(n−1)/n · S / B_link`, crossing over at `S* = n·α_hop·B_link`.
Write `half_size(alpha_s, bw_Bps)` (S½ = α·B) and `per_hop(alpha_s, algbw_Bps, n)` returning
`(alpha_hop, link_Bps)`, and confirm the two views give the same size.

In [ ]:
def half_size(alpha_s: float, bw_Bps: float) -> float:
    return alpha_s * bw_Bps


def per_hop(alpha_s: float, algbw_Bps: float, n: int) -> tuple[float, float]:
    hops = 2 * (n - 1)
    return alpha_s / hops, algbw_Bps * hops / n

In [ ]:
s_half = half_size(alpha, B)
a_hop, link = per_hop(alpha, B, 8)
assert abs(8 * a_hop * link - s_half) < 1e-6 * s_half
assert abs(link - B * 1.75) < 1e-6 * link  # the per-link rate *is* busbw
print(f"✅ S½ = {s_half / 2**20:.1f} MiB: α_hop = {a_hop * 1e6:.2f} µs per step, link {link / 1e9:.0f} GB/s — "
      "busbw is the fitted link bandwidth")

## Regimes, row by row — and an older log format

`AlphaBeta.regime()` labels each size. The second sample (also illustrative) is an `all_gather` log in
the older nccl-tests layout (no `redop`/`root` columns, a float `error` column); the parser handles
both, and the busbw factor is now (n−1)/n.

In [ ]:
model = ab.AlphaBeta(alpha, B)
for r in res.rows[::3]:
    print(f"{r.size:>12} B  {model.regime(r.size):>15}")
legacy = nccltests.parse((FIX / "nccl_all_gather_4gpu_legacy_sample.txt").read_text(), op="all_gather")
s = nccltests.summarize(legacy)
print(f"\nlegacy all_gather over {s['nranks']} ranks: factor {bw.bus_factor('all_gather', s['nranks'])}, "
      f"α = {s['alpha_us']:.1f} µs, busbw -> {s['busbw_asymptote_gbps']:.1f} GB/s, recheck issues: {nccltests.recheck(legacy)}")

## Exercise 4.4 — what tensor parallelism costs a decode step

With the fitted model, one all-reduce of `S` bytes takes `model.time(S)`. Write
`tp_comm_us(model, tokens, hidden, layers, dtype_bytes=2)`: the all-reduce time of one decode step
(two all-reduces of tokens × hidden per layer), in microseconds. Evaluate a 70B-class model (hidden 8192,
80 layers, bf16) at batch 32 on the 8-GPU sample fabric, and compare with a 30 ms inter-token budget.

In [ ]:
def tp_comm_us(model: ab.AlphaBeta, tokens: int, hidden: int, layers: int, dtype_bytes: int = 2) -> float:
    return 2 * layers * model.time(tokens * hidden * dtype_bytes) * 1e6

In [ ]:
msg = 32 * 8192 * 2
t_us = tp_comm_us(model, 32, 8192, 80)
assert abs(t_us - 160 * model.time(msg) * 1e6) < 1e-6
assert model.regime(msg) == "latency-bound"
share = t_us / 30_000
print(f"✅ {msg // 1024} KiB per all-reduce is latency-bound: {t_us / 1e3:.2f} ms of every step "
      f"({share:.0%} of a 30 ms budget); α alone is {160 * model.alpha_s * 1e3:.2f} ms of it")

That is why engines attack **α** for decode: fewer, fused all-reduces; algorithms with fewer steps
(one-shot/two-shot kernels, trees, NVLink SHARP in the switch); overlapping communication with compute;
or a smaller tensor-parallel degree.

## Exercise 4.5 — is the plateau at the link?

Write `diagnose(plateau_busbw_gbps, link_gbps)`: `"at the link"` when the plateau reaches at least 70 %
of the link's per-direction bandwidth, otherwise `"below the link: check the transport"`. Per-direction
figures to compare with (datasheet, verify): PCIe Gen3 x16 ≈ 15.75 GB/s, Gen4 x16 ≈ 31.5, NVLink on
A100 300 GB/s, on H100 450 GB/s.

In [ ]:
def diagnose(plateau_busbw_gbps: float, link_gbps: float) -> str:
    return "at the link" if plateau_busbw_gbps >= 0.7 * link_gbps else "below the link: check the transport"

In [ ]:
assert diagnose(400, 450) == "at the link"  # the sample's plateau vs H100-class NVLink
assert diagnose(5.5, 15.75) == "below the link: check the transport"  # e.g. SHM through host memory on PCIe
print("✅ below the link? NCCL_DEBUG=INFO shows the transport (P2P, SHM, NET); nvidia-smi topo -m shows the path")

## Your own numbers

The same parser reads the tables `gpurt.dist` prints. Here is a quick sweep of this machine's CPU
transport (the notebook 03 backend) — measured, and a very different fabric: the fit tells you its α
and B, and the fit error tells you how well a straight line describes it. Then every nccl-tests or
`gpurt.dist` log you brought back from a GPU box — into the lab's `out/` (`deploy/any-gpu`) or
`deploy/gke/out/` (`run.sh nccl`) — is parsed and summarised the same way.

In [ ]:
from gpurt import env  # noqa: E402
from gpurt.dist.bench import run  # noqa: E402
from gpurt.dist.sweep import format_table  # noqa: E402

backend = "gloo" if env.has_module("torch") else "pipes"
rows = run(backend, "all_reduce", 2, sizes=[8 * 4 ** k for k in range(11)], iters=5, warmup=1)
mine = nccltests.parse(format_table(rows))
print(format_table(rows).splitlines()[1])
print(ab.fit_rows(rows), f"(backend {backend}, measured on this machine)")
print("recheck:", nccltests.recheck(mine) or "consistent")
# logs you brought back from a GPU box: out/ (deploy/any-gpu recipes) or deploy/gke/out/ (run.sh)
for log in sorted(Path("..").glob("out/*.log")) + sorted(Path("..").glob("deploy/*/out/*.log")):
    for run_ in nccltests.parse_many(log.read_text()):
        print(log.name, "(measured)", nccltests.summarize(run_))

## In a design review

**Two minutes.** I read an nccl-tests table by recomputing it: algbw is size over time, busbw is
algbw times 2(n−1)/n for all-reduce — the per-link rate an optimal ring needs — so it compares with the
link whatever the number of GPUs. I fit `t = α + S/B` to the sweep: α is the latency floor, B the
plateau, and S½ = α·B the size where half the bandwidth is reached; below it a faster link does not
help, fewer steps do. I check the plateau against the path — NVLink or PCIe — and if it is well below,
I look at the transport NCCL chose. Then I place the workload on the curve: decode's tensor-parallel
all-reduces are hundreds of kilobytes at most, deep in the latency regime, so their cost is roughly
2 × layers × α per token.

**Drill questions**

1. *busbw is above the NVLink rate. Is the tool wrong?* — No: with NVLink SHARP (NVLS) the switch does the
   reduction, so less data crosses each link than the ring formula assumes. The factor is a convention,
   not a measurement of wire bytes.
2. *Two 8-GPU nodes show 180 GB/s busbw, one node 450. Why?* — Across nodes the ring crosses the NICs; the
   slowest link (network, per-GPU NIC share) bounds it. Hierarchical and tree algorithms help, but the
   plateau is the inter-node bandwidth per GPU.
3. *What single number would you quote for decode all-reduce performance?* — The time at the actual
   message size (e.g. 512 KiB), i.e. the latency floor α — not the plateau busbw at 1 GiB.